In [1]:
import os
import json
import time
import random
from image import process_image
from ocr import run_ocr, get_easyocr_reader
from metrics import score_ocr
from datasets import load_dataset

In [2]:
# point pytesseract at the local Windows install.
# guarded by os.path.exists, so it is a no-op on Chameleon / the container,
# where tesseract is already on PATH with chi_sim beside it.
_tess = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
if os.path.exists(_tess):
    os.environ.setdefault("TESSERACT_CMD", _tess)
    os.environ.setdefault("TESSDATA_PREFIX", r"C:\Users\ishan\tessdata")
print("TESSERACT_CMD =", os.environ.get("TESSERACT_CMD", "(using PATH)"))

TESSERACT_CMD = C:\Program Files\Tesseract-OCR\tesseract.exe


In [3]:
print("--- Loading dataset 'lansinuote/ocr_id_card' ---")
try:
    ds = load_dataset("lansinuote/ocr_id_card", split='train')
    print("Dataset loaded:", len(ds), "images")
except Exception as e:
    print(f"Failed to load dataset: {e}")
    exit()

--- Loading dataset 'lansinuote/ocr_id_card' ---
Dataset loaded: 20000 images


In [4]:
with open("pipeline_configs.json") as f:
    configs = json.load(f)

[c["name"] for c in configs]

['original_tesseract',
 'bright_red_tesseract',
 'contrast_red_green_easyocr',
 'sharpen_blue_easyocr']

In [5]:
# how many images to run. FULL=True runs all 20,000.
SAMPLE_SIZE = 50
SEED = 0
FULL = False

if FULL:
    indices = list(range(len(ds)))
else:
    indices = random.Random(SEED).sample(range(len(ds)), SAMPLE_SIZE)

print("running", len(indices), "images x", len(configs), "configs")

running 50 images x 4 configs


In [6]:
import torch
os.environ["EASYOCR_GPU"] = "1" if torch.cuda.is_available() else "0"
print("EASYOCR_GPU =", os.environ["EASYOCR_GPU"])

EASYOCR_GPU = 1


In [7]:
# build the EasyOCR reader once here, so its startup is not counted in any config's time
t = time.perf_counter()
get_easyocr_reader()
print(f"easyocr ready in {time.perf_counter() - t:.1f}s")

easyocr ready in 1.5s


In [8]:
rows = []
for config in configs:
    n_fields = 0
    n_exact = 0
    per_class = {}
    start = time.perf_counter()
    for i in indices:
        row = ds[i]
        processed = process_image(row["image"], config)
        text = run_ocr(processed, config["ocr"])
        result = score_ocr(text, row["ocr"])
        n_fields += result["n_fields"]
        n_exact += result["n_exact"]
        for field, counts in result["per_class"].items():
            acc = per_class.setdefault(field, [0, 0])
            acc[0] += counts[0]
            acc[1] += counts[1]
    elapsed = time.perf_counter() - start
    rows.append({
        "name": config["name"],
        "engine": config["ocr"],
        "recall": n_exact / n_fields if n_fields else 0.0,
        "mean_latency": elapsed / len(indices),
        "total_s": elapsed,
        "per_class": per_class,
    })
    print(f'{config["name"]:<28} recall={rows[-1]["recall"]:.3f}  {rows[-1]["mean_latency"]:.3f}s/img')

original_tesseract           recall=0.492  0.725s/img
bright_red_tesseract         recall=0.495  0.703s/img
contrast_red_green_easyocr   recall=0.745  0.264s/img
sharpen_blue_easyocr         recall=0.690  0.177s/img


In [9]:
print(f'{"config":<28}{"engine":<10}{"recall":>8}{"s/img":>9}{"total_s":>9}')
for r in rows:
    print(f'{r["name"]:<28}{r["engine"]:<10}{r["recall"]:>8.3f}{r["mean_latency"]:>9.3f}{r["total_s"]:>9.1f}')

config                      engine      recall    s/img  total_s
original_tesseract          tesseract    0.492    0.725     36.2
bright_red_tesseract        tesseract    0.495    0.703     35.1
contrast_red_green_easyocr  easyocr      0.745    0.264     13.2
sharpen_blue_easyocr        easyocr      0.690    0.177      8.9


In [10]:
fields = ["name", "gender", "ethnicity", "birth_year", "birth_month", "birth_day", "address", "id_number"]
for r in rows:
    parts = []
    for field in fields:
        hits, total = r["per_class"].get(field, [0, 0])
        parts.append(f"{field} {hits}/{total}")
    print(r["name"] + ": " + ", ".join(parts))

original_tesseract: name 11/50, gender 16/50, ethnicity 9/50, birth_year 40/50, birth_month 40/50, birth_day 44/50, address 10/50, id_number 27/50
bright_red_tesseract: name 11/50, gender 17/50, ethnicity 15/50, birth_year 38/50, birth_month 40/50, birth_day 40/50, address 11/50, id_number 26/50
contrast_red_green_easyocr: name 26/50, gender 48/50, ethnicity 31/50, birth_year 46/50, birth_month 48/50, birth_day 49/50, address 15/50, id_number 35/50
sharpen_blue_easyocr: name 28/50, gender 43/50, ethnicity 16/50, birth_year 47/50, birth_month 48/50, birth_day 48/50, address 15/50, id_number 31/50
